# ReparoS Base V2-32 — complete offline experiment

Train a complete Base V2 from scratch: a new train-only SentencePiece tokenizer, new OpenNMT vocabularies and new model weights. The original Base keeps its original tokenizer during evaluation. Everything runs from the attached offline Kaggle Dataset.


## 1. Configuration

Attach three Kaggle datasets before running: Base V2 data, the original Base artifact, and the curriculum/evaluation dataset. Paths are discovered from their required files, so the Kaggle dataset slug may change.


In [ ]:
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/QU-solution')
RUN_ROOT = Path('/kaggle/working/reparos/base-v2-32-tokenizer-v2-run')
KAGGLE_INPUT = Path('/kaggle/input')

TRAIN_STEPS = 10_000
VALID_STEPS = 1_000
CHECKPOINT_STEPS = 1_000
BATCH_SIZE_TOKENS = 131_072
BUCKET_SIZE = 262_144
NUM_WORKERS = 8
RESUME_IF_AVAILABLE = True
SEED = 2026
TOKENIZER_CORPUS_SIZE = 500_000
TOKENIZER_VOCAB_SIZE = 8_000


## 2. Offline runtime setup

In [ ]:
import json, os, shutil, subprocess, sys

# Fixed path of the attached private Kaggle Dataset.
BUNDLE_ROOT = Path('/kaggle/input/datasets/vanhieu1125/reparos-haha/kaggle-bundle')
required_bundle_dirs = ('project', 'wheels', 'data', 'artifacts', 'evaluation')
assert all((BUNDLE_ROOT / name).is_dir() for name in required_bundle_dirs), BUNDLE_ROOT

# Copy source out of read-only /kaggle/input.
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
shutil.copytree(BUNDLE_ROOT / 'project', REPO_ROOT)

# Install only bundled Linux CPython 3.12 wheels. Torch is deliberately absent:
# keep Kaggle's CUDA-enabled torch 2.10.0+cu128 for Blackwell sm_120.
SITE_PACKAGES = Path('/kaggle/working/site-packages')
if SITE_PACKAGES.exists():
    shutil.rmtree(SITE_PACKAGES)
SITE_PACKAGES.mkdir(parents=True)
wheels = sorted((BUNDLE_ROOT / 'wheels').glob('*.whl'))
assert wheels, BUNDLE_ROOT / 'wheels'
assert not any(path.name.startswith(('torch-', 'nvidia_')) for path in wheels)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--no-index', '--no-deps', '--target', str(SITE_PACKAGES),
    *map(str, wheels),
], check=True)

# CTranslate2 4.6.x does not expose the 4.8.x unsafe_deserialization
# constructor argument. Checkpoint trust is already explicitly required by CLI.
ct2_source = REPO_ROOT / 'src/reparos/serving/ctranslate2.py'
ct2_text = ct2_source.read_text(encoding='utf-8')
ct2_old = """    converter = ctranslate2.converters.OpenNMTPyConverter(
        str(checkpoint_path), unsafe_deserialization=trust_checkpoint,
    )
"""
ct2_new = """    try:
        converter = ctranslate2.converters.OpenNMTPyConverter(
            str(checkpoint_path), unsafe_deserialization=trust_checkpoint,
        )
    except TypeError as error:
        if 'unsafe_deserialization' not in str(error):
            raise
        if not trust_checkpoint:
            raise RuntimeError(
                'CTranslate2 4.6.x requires --trust-checkpoint for OpenNMT conversion'
            ) from error
        converter = ctranslate2.converters.OpenNMTPyConverter(str(checkpoint_path))
"""
if ct2_old in ct2_text:
    ct2_source.write_text(ct2_text.replace(ct2_old, ct2_new), encoding='utf-8')

os.chdir(REPO_ROOT)
SRC_ROOT = REPO_ROOT / 'src'
python_path = os.pathsep.join((str(SRC_ROOT), str(SITE_PACKAGES)))
os.environ['PYTHONPATH'] = python_path + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path[:0] = [str(SRC_ROOT), str(SITE_PACKAGES)]

# Helpers expect executable CLI paths. Small local wrappers invoke the repo
# modules with the Kaggle Python interpreter and inherited PYTHONPATH.
BIN_ROOT = Path('/kaggle/working/qu-solution-bin')
BIN_ROOT.mkdir(parents=True, exist_ok=True)
VENV_PY = Path(sys.executable)
REPAROS = BIN_ROOT / 'reparos'
BENCHMARK = BIN_ROOT / 'benchmark-three'
REPAROS.write_text(f'#!{sys.executable}\nfrom reparos.cli import main\nmain()\n', encoding='utf-8')
BENCHMARK.write_text(f'#!{sys.executable}\nfrom benchmarking.cli import main\nmain()\n', encoding='utf-8')
REPAROS.chmod(0o755)
BENCHMARK.chmod(0o755)

# All subsequent artifact discovery must search this bundle, not every attached
# Kaggle Dataset.
KAGGLE_INPUT = BUNDLE_ROOT

import torch, onmt, pyonmttok, sentencepiece, ctranslate2
assert torch.cuda.is_available(), 'GPU accelerator is not enabled'
print('Bundle:', BUNDLE_ROOT)
print('Python:', sys.version)
print('Torch:', torch.__version__, torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print('OpenNMT:', getattr(onmt, '__version__', 'unknown'))
print('CTranslate2:', ctranslate2.__version__)


## 3. Locate and validate attached artifacts

In [ ]:
def unique_match(pattern, label, predicate=lambda path: True):
    matches = [path for path in KAGGLE_INPUT.rglob(pattern) if predicate(path)]
    assert len(matches) == 1, f'{label}: expected exactly one match, found {matches}'
    return matches[0]

# Base V2 root is the directory containing train.noisy.src.
BASE_V2_ROOT = unique_match(
    'train.noisy.src', 'Base V2 train data',
    lambda p: (p.parent / 'train.clean.src').is_file() and (p.parent / 'validation.noisy.src').is_file(),
).parent

# Original Base artifact supplies the frozen tokenizer/vocabulary, decoding config,
# and the Base checkpoint used only as an evaluation baseline.
ORIGINAL_CONFIG = unique_match(
    'opennmt-base.json', 'original Base config',
    lambda p: list(p.parent.glob('reparos_base_step_*.pt')) and (p.parent / 'vocab.src').is_file(),
)
BASE_ARTIFACT_DIR = ORIGINAL_CONFIG.parent
ORIGINAL_TOKENIZER_MODEL = unique_match('tokenizer.model', 'original tokenizer')
DECODING_CONFIG = BASE_ARTIFACT_DIR / 'decoding-config.json'

USER_GOLD = unique_match('user-centric-v2.jsonl', 'User-centric 88')
EVALUATION_ROOT = USER_GOLD.parent
for name in ('composition-4k.jsonl', 'diagnostic-10k.jsonl'):
    assert (EVALUATION_ROOT / name).is_file(), EVALUATION_ROOT / name

MANIFEST = BASE_V2_ROOT.parents[1] / 'base-v2-manifest.json'
assert MANIFEST.is_file(), MANIFEST
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
assert manifest['leakage'] == {'train_validation': 0, 'train_test': 0, 'validation_test': 0}

required = []
for split in ('train', 'validation', 'test'):
    for lane in ('noisy', 'clean'):
        for suffix in ('src', 'tgt', 'meta.jsonl'):
            required.append(BASE_V2_ROOT / f'{split}.{lane}.{suffix}')
assert all(path.is_file() for path in required), [str(p) for p in required if not p.is_file()]

print('Base V2:', BASE_V2_ROOT)
print('Manifest:', MANIFEST)
print('Original Base:', BASE_ARTIFACT_DIR)
print('Original tokenizer:', ORIGINAL_TOKENIZER_MODEL)
print('Evaluation:', EVALUATION_ROOT)


## 4. Train a new tokenizer, build new vocabularies and configure Base V2

The tokenizer corpus contains every accepted clean training query plus a deterministic reservoir sample of noisy training queries, capped at 500K total lines. Validation and test are never used. OpenNMT vocabularies are then rebuilt from Base V2 with this tokenizer.


In [ ]:
import random
from reparos.config import TokenizerConfig
from reparos.tokenization import train_sentencepiece

RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Construct a balanced, deterministic train-only tokenizer corpus.
clean_lines = (BASE_V2_ROOT / 'train.clean.src').read_text(encoding='utf-8').splitlines()
noisy_budget = max(0, TOKENIZER_CORPUS_SIZE - len(clean_lines))
rng = random.Random(SEED)
noisy_sample = []
with (BASE_V2_ROOT / 'train.noisy.src').open(encoding='utf-8') as stream:
    for index, line in enumerate(stream):
        value = line.rstrip('\n\r')
        if index < noisy_budget:
            noisy_sample.append(value)
        else:
            replacement = rng.randint(0, index)
            if replacement < noisy_budget:
                noisy_sample[replacement] = value
tokenizer_lines = clean_lines + noisy_sample
rng.shuffle(tokenizer_lines)
TOKENIZER_CORPUS = RUN_ROOT / 'tokenizer-train.txt'
TOKENIZER_CORPUS.write_text('\n'.join(tokenizer_lines) + '\n', encoding='utf-8')
assert len(tokenizer_lines) == min(TOKENIZER_CORPUS_SIZE, len(clean_lines) + 5_410_493)

TOKENIZER_ROOT = RUN_ROOT / 'tokenizer-v2'
train_sentencepiece(
    [TOKENIZER_CORPUS], TOKENIZER_ROOT,
    TokenizerConfig(vocab_size=TOKENIZER_VOCAB_SIZE, input_sentence_size=0),
)
TOKENIZER_MODEL = TOKENIZER_ROOT / 'tokenizer.model'
assert TOKENIZER_MODEL.is_file()

def concatenate(destination, sources):
    with destination.open('wb') as target:
        for source in sources:
            with source.open('rb') as stream:
                shutil.copyfileobj(stream, target, length=16 * 1024 * 1024)

VALID_SRC = RUN_ROOT / 'validation.src'
VALID_TGT = RUN_ROOT / 'validation.tgt'
concatenate(VALID_SRC, [BASE_V2_ROOT / 'validation.noisy.src', BASE_V2_ROOT / 'validation.clean.src'])
concatenate(VALID_TGT, [BASE_V2_ROOT / 'validation.noisy.tgt', BASE_V2_ROOT / 'validation.clean.tgt'])

config = json.loads(ORIGINAL_CONFIG.read_text(encoding='utf-8'))
config.update({
    'save_data': str(RUN_ROOT / 'vocab'),
    'src_vocab': str(RUN_ROOT / 'vocab.src'),
    'tgt_vocab': str(RUN_ROOT / 'vocab.tgt'),
    'src_vocab_size': TOKENIZER_VOCAB_SIZE,
    'tgt_vocab_size': TOKENIZER_VOCAB_SIZE,
    'src_subword_model': str(TOKENIZER_MODEL),
    'tgt_subword_model': str(TOKENIZER_MODEL),
    'save_model': str(RUN_ROOT / 'reparos_base_v2'),
    'train_steps': TRAIN_STEPS,
    'valid_steps': VALID_STEPS,
    'save_checkpoint_steps': CHECKPOINT_STEPS,
    'batch_size': BATCH_SIZE_TOKENS,
    'bucket_size': BUCKET_SIZE,
    'num_workers': NUM_WORKERS,
    'keep_checkpoint': 10,
    'seed': SEED,
    'data': {
        'noisy': {
            'path_src': str(BASE_V2_ROOT / 'train.noisy.src'),
            'path_tgt': str(BASE_V2_ROOT / 'train.noisy.tgt'),
            'transforms': ['sentencepiece'], 'weight': 75,
        },
        'clean': {
            'path_src': str(BASE_V2_ROOT / 'train.clean.src'),
            'path_tgt': str(BASE_V2_ROOT / 'train.clean.tgt'),
            'transforms': ['sentencepiece'], 'weight': 25,
        },
        'valid': {
            'path_src': str(VALID_SRC), 'path_tgt': str(VALID_TGT),
            'transforms': ['sentencepiece'],
        },
    },
})
config.pop('train_from', None)

def checkpoint_step(path):
    return int(path.stem.rsplit('_step_', 1)[1])

existing = sorted(RUN_ROOT.glob('reparos_base_v2_step_*.pt'), key=checkpoint_step)
if existing and RESUME_IF_AVAILABLE:
    config['train_from'] = str(existing[-1])
    assert checkpoint_step(existing[-1]) < TRAIN_STEPS, (existing[-1], TRAIN_STEPS)
    print('Resuming from:', existing[-1])
elif existing:
    raise FileExistsError(f'Existing checkpoints found: {existing}')
else:
    print('Fresh Base V2 training with a new tokenizer and vocabulary')

CONFIG_PATH = RUN_ROOT / 'opennmt-base-v2.json'
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

# A fresh run must build OpenNMT token-frequency vocabularies with the new tokenizer.
if not existing:
    subprocess.run([
        str(VENV_PY), '-m', 'onmt.bin.build_vocab',
        '-config', str(CONFIG_PATH), '-n_sample', '-1',
    ], check=True)
assert (RUN_ROOT / 'vocab.src').is_file() and (RUN_ROOT / 'vocab.tgt').is_file()
print('Tokenizer:', TOKENIZER_MODEL)
print('Tokenizer corpus rows:', len(tokenizer_lines))
print('Config:', CONFIG_PATH)


## 5. Train Base V2-32

In [ ]:
subprocess.run([
    str(VENV_PY), '-m', 'onmt.bin.train', '-config', str(CONFIG_PATH)
], check=True)

checkpoints = sorted(RUN_ROOT.glob('reparos_base_v2_step_*.pt'), key=checkpoint_step)
assert checkpoints, RUN_ROOT
BASE_V2_CHECKPOINT = checkpoints[-1]
assert checkpoint_step(BASE_V2_CHECKPOINT) == TRAIN_STEPS, BASE_V2_CHECKPOINT
print('Base V2 checkpoint:', BASE_V2_CHECKPOINT)


## 6. Evaluate original Base and Base V2 in the same session

Both checkpoints use the same tokenizer, decoding configuration, CTranslate2 conversion and three frozen benchmarks. This cell may be rerun without retraining.


In [ ]:
from reparos.curriculum_notebook import evaluate_checkpoint
from reparos.curriculum_training import latest_checkpoint

ORIGINAL_BASE_CHECKPOINT = latest_checkpoint(BASE_ARTIFACT_DIR, 'reparos_base')

reports = {}
for system, checkpoint, tokenizer in (
    ('base', ORIGINAL_BASE_CHECKPOINT, ORIGINAL_TOKENIZER_MODEL),
    ('base_v2_32', BASE_V2_CHECKPOINT, TOKENIZER_MODEL),
):
    reports[system] = evaluate_checkpoint(
        checkpoint=checkpoint,
        tokenizer=tokenizer,
        decoding_config=DECODING_CONFIG,
        evaluation_root=EVALUATION_ROOT,
        output_root=RUN_ROOT / 'evaluation' / system,
        reparos_cli=REPAROS,
        benchmark_cli=BENCHMARK,
        device='cuda',
        compute_type='float16',
    )

REPORT_PATH = RUN_ROOT / 'base-v2-comparison.json'
REPORT_PATH.write_text(json.dumps(reports, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(reports, ensure_ascii=False, indent=2))


## 7. Show metric deltas

In [ ]:
def flatten_numbers(value, prefix=''):
    result = {}
    if isinstance(value, dict):
        for key, child in value.items():
            name = f'{prefix}.{key}' if prefix else key
            result.update(flatten_numbers(child, name))
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        result[prefix] = float(value)
    return result

rows = []
for benchmark in ('user', 'composition', 'diagnostic'):
    old = flatten_numbers(reports['base'][benchmark])
    new = flatten_numbers(reports['base_v2_32'][benchmark])
    for metric in sorted(old.keys() & new.keys()):
        rows.append({
            'benchmark': benchmark,
            'metric': metric,
            'base': old[metric],
            'base_v2_32': new[metric],
            'delta': new[metric] - old[metric],
        })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    print(json.dumps(rows, ensure_ascii=False, indent=2))


## 8. Inspect improvements and regressions

In [ ]:
def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

case_rows = []
for benchmark in ('user', 'composition', 'diagnostic'):
    base_rows = {
        row['query_id']: row
        for row in load_jsonl(RUN_ROOT / 'evaluation' / 'base' / f'{benchmark}.predictions.jsonl')
    }
    v2_rows = load_jsonl(RUN_ROOT / 'evaluation' / 'base_v2_32' / f'{benchmark}.predictions.jsonl')
    for new in v2_rows:
        old = base_rows[new['query_id']]
        old_ok = old['output'] == old['expected']
        new_ok = new['output'] == new['expected']
        if old_ok != new_ok:
            case_rows.append({
                'status': 'IMPROVED' if new_ok else 'REGRESSED',
                'benchmark': benchmark,
                'error_type': new['error_type'],
                'input': new['input'],
                'expected': new['expected'],
                'base': old['output'],
                'base_v2_32': new['output'],
            })

case_rows.sort(key=lambda row: (row['status'] != 'IMPROVED', row['benchmark'], row['error_type']))
CASE_PATH = RUN_ROOT / 'base-v2-cases.json'
CASE_PATH.write_text(json.dumps(case_rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
try:
    import pandas as pd
    frame = pd.DataFrame(case_rows)
    print(frame['status'].value_counts())
    display(frame.groupby(['status', 'benchmark']).head(20))
except ImportError:
    print(json.dumps(case_rows[:120], ensure_ascii=False, indent=2))


## 9. Package checkpoint, CT2 model, predictions and reports

In [ ]:
EXPORT_ROOT = RUN_ROOT / 'final-artifact'
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)

shutil.copy2(BASE_V2_CHECKPOINT, EXPORT_ROOT / BASE_V2_CHECKPOINT.name)
shutil.copy2(TOKENIZER_MODEL, EXPORT_ROOT / 'tokenizer.model')
shutil.copy2(TOKENIZER_ROOT / 'tokenizer.vocab', EXPORT_ROOT / 'tokenizer.vocab')
shutil.copy2(TOKENIZER_ROOT / 'tokenizer-manifest.json', EXPORT_ROOT / 'tokenizer-manifest.json')
shutil.copy2(RUN_ROOT / 'vocab.src', EXPORT_ROOT / 'vocab.src')
shutil.copy2(RUN_ROOT / 'vocab.tgt', EXPORT_ROOT / 'vocab.tgt')
shutil.copy2(DECODING_CONFIG, EXPORT_ROOT / 'decoding-config.json')
shutil.copy2(CONFIG_PATH, EXPORT_ROOT / 'opennmt-base-v2.json')
shutil.copy2(MANIFEST, EXPORT_ROOT / 'base-v2-manifest.json')
shutil.copy2(REPORT_PATH, EXPORT_ROOT / REPORT_PATH.name)
shutil.copy2(CASE_PATH, EXPORT_ROOT / CASE_PATH.name)
shutil.copytree(RUN_ROOT / 'evaluation/base_v2_32/ctranslate2', EXPORT_ROOT / 'ctranslate2')
shutil.copytree(RUN_ROOT / 'evaluation', EXPORT_ROOT / 'evaluation')

zip_path = Path(shutil.make_archive('/kaggle/working/reparos-base-v2-32-final', 'zip', EXPORT_ROOT))
print('DOWNLOAD:', zip_path, f'({zip_path.stat().st_size / 1024**2:.1f} MiB)')
print('Comparison report:', REPORT_PATH)
